# Database Visualization

Visualizing data is always nice. In larger projects you could set up a dashboard such as Grafana.
For this project we'll stick to simpler visualizations using widgets in jupyter notebook.
You don't have to write or update any code in this notebook, it just serves as an example.

It reuses the helper functions you wrote in `0-sqlite-primer.ipynb`.

We develop three visualizations:
- ship roll + draft
- wharf occupancy grid
- ship occupancy grid

Then in the end, those three visualizations are compbined in a widget with a little dropdown menu that allows you to select the ship you'd like to visualize.
The widget also has a background thread the moves around containers and randomly changes the ships' roll and draft, such that you can see the data updating in real time.

A separate database file is used so this notebook does not interfere with the previous exercise.

## Importing the helper functions

This first cell makes the helper functions from the other notebook available in this one.

In [1]:
import ast
import json
import sqlite3
import time
from pathlib import Path
from IPython.display import clear_output, display

PRIMER_NOTEBOOK_PATH = Path('0-sqlite-primer.ipynb')
VIS_DB_PATH = Path('harbour-system-visualization.sqlite')

PRIMER_SYMBOLS = {
    # schema query strings used by create_schema
    'create_container_table_query',
    'create_ship_table_query',
    'create_wharf_slot_table_query',
    'create_ship_slot_table_query',
    'create_wharf_slot_cross_presence_trigger_query',
    'create_ship_slot_cross_presence_trigger_query',
    # helper functions from notebook 0
    'create_schema',
    'seed_wharf_slots',
    'seed_ship_slots',
    'intake_container',
    'place_container_in_wharf_slot',
    'add_ship',
    'load_container_onto_ship',
    'list_wharf_state',
    'list_ship_state',
    'update_ship_state',
    'find_container',
}

def _cell_defines_symbol(source, target_symbols):
    try:
        tree = ast.parse(source)
    except SyntaxError:
        return False

    for node in tree.body:
        if isinstance(node, ast.FunctionDef) and node.name in target_symbols:
            return True
        if isinstance(node, (ast.Assign, ast.AnnAssign)):
            targets = []
            if isinstance(node, ast.Assign):
                targets = node.targets
            else:
                targets = [node.target]
            for target in targets:
                if isinstance(target, ast.Name) and target.id in target_symbols:
                    return True
    return False

def load_primer_helpers(primer_path=PRIMER_NOTEBOOK_PATH, target_symbols=PRIMER_SYMBOLS):
    if not primer_path.exists():
        raise FileNotFoundError(f'Could not find primer notebook at {primer_path.resolve()}')

    with primer_path.open('r', encoding='utf-8') as handle:
        notebook_data = json.load(handle)

    for cell in notebook_data.get('cells', []):
        if cell.get('cell_type') != 'code':
            continue

        source = ''.join(cell.get('source', []))
        if _cell_defines_symbol(source, target_symbols):
            exec(source, globals())

    missing = [name for name in target_symbols if name not in globals()]
    if missing:
        raise RuntimeError(f'Missing expected helper symbols from primer notebook: {missing}')

def get_vis_connection(db_path=VIS_DB_PATH):
    connection = sqlite3.connect(db_path)
    connection.row_factory = sqlite3.Row
    connection.execute('PRAGMA foreign_keys = ON;')
    return connection

load_primer_helpers()
print('✅ Primer helper functions loaded')

✅ Primer helper functions loaded


## Initialize a New Database

Run this cell to create a clean demo state in `harbour-system-visualization.sqlite`.

In [2]:
def initialize_visualization_database(db_path=VIS_DB_PATH):
    if db_path.exists():
        db_path.unlink()

    with get_vis_connection(db_path) as connection:
        create_schema(connection)

        # Wharf grid (3 x 4)
        seed_wharf_slots(connection, 3, 4)

        # Two ships with different dimensions
        add_ship(connection, 1, roll_deg=0.6, draft_m=6.8)
        add_ship(connection, 2, roll_deg=2.1, draft_m=7.4)
        seed_ship_slots(connection, ship_id=1, rows=2, cols=3)
        seed_ship_slots(connection, ship_id=2, rows=1, cols=4)

        # Intake some containers
        for container_id, weight_kg in [
            (100, 10150),
            (101, 9850),
            (102, 11020),
            (103, 9600),
            (104, 12010),
            (105, 10890),
        ]:
            intake_container(connection, container_id, weight_kg)

        # Put containers on wharf
        place_container_in_wharf_slot(connection, 100, 1)
        place_container_in_wharf_slot(connection, 101, 2)
        place_container_in_wharf_slot(connection, 102, 3)
        place_container_in_wharf_slot(connection, 103, 5)
        place_container_in_wharf_slot(connection, 104, 8)
        place_container_in_wharf_slot(connection, 105, 10)

        # Move two containers to ships
        load_container_onto_ship(connection, 101, ship_id=1, slot_id=1)
        load_container_onto_ship(connection, 104, ship_id=2, slot_id=2)

        update_ship_state(connection, ship_id=1, roll_deg=1.1, draft_m=7.0)
        update_ship_state(connection, ship_id=2, roll_deg=2.6, draft_m=7.9)

    print(f'✅ Visualization DB initialized at: {db_path}')

initialize_visualization_database()

✅ Visualization DB initialized at: harbour-system-visualization.sqlite


## 1. Ship roll/draft visualization

Shows the ship's roll and draft.
Roll is shown like a bubble gauge (such as you find on a level-gauge).
Draft is just shown as a loading bar.

Set `SELECTED_SHIP_ID` and rerun the cell.

Use `watch_ship_attitude(...)` for continuous refresh.

In [3]:
def list_ship_ids(db_path=VIS_DB_PATH):
    with get_vis_connection(db_path) as connection:
        rows = connection.execute('SELECT ship_id FROM ship ORDER BY ship_id').fetchall()
        return [row['ship_id'] for row in rows]


def _gauge(value, min_value, max_value, width=20):
    if max_value <= min_value:
        return '-' * width

    clipped = max(min_value, min(max_value, value))
    ratio = (clipped - min_value) / (max_value - min_value)
    filled = int(round(ratio * width))
    return '█' * filled + '-' * (width - filled)


def _bubble_level(value, min_value=-15, max_value=15, width=13):
    if width < 3:
        width = 3

    if max_value <= min_value:
        center_index = width // 2
        chars = ['-'] * width
        chars[center_index] = '|'
        chars[0] = 'o'
        return '|' + ''.join(chars) + '|'

    clipped = max(min_value, min(max_value, value))
    ratio = (clipped - min_value) / (max_value - min_value)
    bubble_index = int(round(ratio * (width - 1)))
    center_index = width // 2

    chars = ['-'] * width
    chars[bubble_index] = 'o'
    if bubble_index != center_index:
        chars[center_index] = '|'

    return '|' + ''.join(chars) + '|'


def render_ship_attitude(ship_id, db_path=VIS_DB_PATH):
    with get_vis_connection(db_path) as connection:
        ship = connection.execute(
            'SELECT ship_id, roll_deg, draft_m FROM ship WHERE ship_id = ?',
            (ship_id,),
        ).fetchone()

    if ship is None:
        print(f'Ship {ship_id} not found.')
        return

    roll = ship['roll_deg']
    draft = ship['draft_m']

    attitude_width = 21
    print(f'Ship {ship_id} attitude')
    print(f'Roll  (deg): {roll:6.2f} {_bubble_level(roll, -15, 15, width=attitude_width)} range [-15, 15]')
    print(f'Draft (m)  : {draft:6.2f} |{_gauge(draft, 0, 12, width=attitude_width)}| range [0, 12]')


def watch_ship_attitude(ship_id, interval_s=2.0, iterations=30, db_path=VIS_DB_PATH):
    for _ in range(iterations):
        clear_output(wait=True)
        render_ship_attitude(ship_id, db_path=db_path)
        print(f'Auto-refresh every {interval_s}s - Ctrl/Cmd+M I to interrupt')
        time.sleep(interval_s)


print('Available ships:', list_ship_ids())
SELECTED_SHIP_ID = 2
render_ship_attitude(SELECTED_SHIP_ID)

Available ships: [1, 2]
Ship 2 attitude
Roll  (deg):   2.60 |----------|-o--------| range [-15, 15]
Draft (m)  :   7.90 |██████████████-------| range [0, 12]


## 2. Wharf occupancy grid visualization

Run this cell to render the full wharf grid (`☑` = occupied, `☐` = empty).

Use `watch_wharf_grid(...)` for continuous refresh.

In [4]:
def render_wharf_grid(db_path=VIS_DB_PATH):

    with get_vis_connection(db_path) as connection:
        state = list_wharf_state(connection)

    if not state:
        print('Wharf is empty (no slots found).')
        return

    max_row = max(slot['row'] for slot in state)
    max_col = max(slot['col'] for slot in state)
    by_position = {(slot['row'], slot['col']): slot for slot in state}

    occupied_marker = '☑'
    empty_marker = '☐'

    print('Wharf occupancy')

    for row in range(1, max_row + 1):
        line = []
        for col in range(1, max_col + 1):
            slot = by_position.get((row, col))
            if slot is None:
                line.append(' ')
            elif slot['occupancy'] == 'occupied':
                line.append(occupied_marker)
            else:
                line.append(empty_marker)

        print(f'row {row}: ' + ' '.join(line))

def watch_wharf_grid(interval_s=2.0, iterations=30, db_path=VIS_DB_PATH):
    for _ in range(iterations):
        clear_output(wait=True)
        render_wharf_grid(db_path=db_path)
        print(f'Auto-refresh every {interval_s}s - Ctrl/Cmd+M I to interrupt')
        time.sleep(interval_s)

render_wharf_grid()

Wharf occupancy
row 1: ☑ ☐ ☑ ☐
row 2: ☑ ☐ ☐ ☐
row 3: ☐ ☑ ☐ ☐


## 3. Ship occupancy grid visualization

Same but for the ship's grid.

Set `SELECTED_SHIP_ID` and rerun, or use `watch_ship_grid(...)` for continuous refresh.

In [5]:
def render_ship_grid(ship_id, db_path=VIS_DB_PATH):
    with get_vis_connection(db_path) as connection:
        state = list_ship_state(connection, ship_id)

    if not state:
        print(f'No ship slots found for ship_id={ship_id}.')
        return

    max_row = max(slot['row'] for slot in state)
    max_col = max(slot['col'] for slot in state)
    by_position = {(slot['row'], slot['col']): slot for slot in state}

    occupied_marker = '☑'
    empty_marker = '☐'

    print(f'Ship {ship_id} occupancy')

    for row in range(1, max_row + 1):
        line = []
        for col in range(1, max_col + 1):
            slot = by_position.get((row, col))
            if slot is None:
                line.append(' ')
            elif slot['occupancy'] == 'occupied':
                line.append(occupied_marker)
            else:
                line.append(empty_marker)

        print(f'row {row}: ' + ' '.join(line))

def watch_ship_grid(ship_id, interval_s=2.0, iterations=30, db_path=VIS_DB_PATH):
    for _ in range(iterations):
        clear_output(wait=True)
        render_ship_grid(ship_id, db_path=db_path)
        print(f'Auto-refresh every {interval_s}s - Ctrl/Cmd+M I to interrupt')
        time.sleep(interval_s)

SELECTED_SHIP_ID = 1
render_ship_grid(SELECTED_SHIP_ID)

Ship 1 occupancy
row 1: ☑ ☐ ☐
row 2: ☐ ☐ ☐


## Live updating widget

This first cell defines a couple of functions that load/unload a ship and randomly changes the roll and draft.
Runnign this in a background thread allows you to update the data in the database in the background, while the visualization is running.

In [6]:
LIVE_DEMO_STATE = {'thread': None, 'stop_event': None, 'phase': 'unloading'}

def stop_live_demo_writer():
    thread = LIVE_DEMO_STATE.get('thread')
    stop_event = LIVE_DEMO_STATE.get('stop_event')

    if stop_event is not None:
        stop_event.set()
    if thread is not None and thread.is_alive():
        thread.join(timeout=1.0)

    LIVE_DEMO_STATE['thread'] = None
    LIVE_DEMO_STATE['stop_event'] = None
    LIVE_DEMO_STATE['phase'] = 'unloading'
    print('Live demo writer stopped.')

def _move_one_container_from_wharf_to_ship(connection, ship_id):
    source_container = connection.execute(
        """
        SELECT slot_id, container_id
        FROM wharf_slot
        WHERE occupancy = 'occupied'
        ORDER BY slot_id
        LIMIT 1
        """
    ).fetchone()

    target_slot = connection.execute(
        """
        SELECT slot_id
        FROM ship_slot
        WHERE ship_id = ? AND occupancy = 'empty'
        ORDER BY slot_id
        LIMIT 1
        """,
        (ship_id,),
    ).fetchone()

    if source_container is None or target_slot is None:
        return False

    load_container_onto_ship(
        connection,
        source_container['container_id'],
        ship_id=ship_id,
        slot_id=target_slot['slot_id'],
    )
    return True

def _move_one_container_from_ship_to_wharf(connection, ship_id):
    source_container = connection.execute(
        """
        SELECT slot_id, container_id
        FROM ship_slot
        WHERE ship_id = ? AND occupancy = 'occupied'
        ORDER BY slot_id
        LIMIT 1
        """,
        (ship_id,),
    ).fetchone()

    target_slot = connection.execute(
        """
        SELECT slot_id
        FROM wharf_slot
        WHERE occupancy = 'empty'
        ORDER BY slot_id
        LIMIT 1
        """
    ).fetchone()

    if source_container is None or target_slot is None:
        return False

    container_id = source_container['container_id']
    ship_slot_id = source_container['slot_id']
    wharf_slot_id = target_slot['slot_id']

    ship_clear = connection.execute(
        """
        UPDATE ship_slot
        SET occupancy = 'empty', container_id = NULL
        WHERE ship_id = ?
          AND slot_id = ?
          AND occupancy = 'occupied'
          AND container_id = ?
        """,
        (ship_id, ship_slot_id, container_id),
    )
    if ship_clear.rowcount == 0:
        return False

    wharf_fill = connection.execute(
        """
        UPDATE wharf_slot
        SET occupancy = 'occupied', container_id = ?
        WHERE slot_id = ?
          AND occupancy = 'empty'
        """,
        (container_id, wharf_slot_id),
    )
    if wharf_fill.rowcount == 0:
        connection.execute(
            """
            UPDATE ship_slot
            SET occupancy = 'occupied', container_id = ?
            WHERE ship_id = ?
              AND slot_id = ?
              AND occupancy = 'empty'
            """,
            (container_id, ship_id, ship_slot_id),
        )
        return False

    return True

def start_live_demo_writer(ship_id=1, update_interval_s=0.5, db_path=VIS_DB_PATH):
    stop_live_demo_writer()

    stop_event = threading.Event()
    LIVE_DEMO_STATE['phase'] = 'unloading'

    def _writer_loop():
        while not stop_event.is_set():
            try:
                with get_vis_connection(db_path) as connection:
                    ship = connection.execute(
                        'SELECT roll_deg, draft_m FROM ship WHERE ship_id = ?',
                        (ship_id,),
                    ).fetchone()

                    if ship is not None:
                        roll_delta = random.choice([-0.5, -0.3, 0.3, 0.5])
                        draft_delta = random.choice([-0.1, 0.1])
                        new_roll = max(-15.0, min(15.0, ship['roll_deg'] + roll_delta))
                        new_draft = max(0.0, min(12.0, ship['draft_m'] + draft_delta))
                        update_ship_state(
                            connection,
                            ship_id=ship_id,
                            roll_deg=round(new_roll, 2),
                            draft_m=round(new_draft, 2),
                        )

                        phase = LIVE_DEMO_STATE.get('phase', 'unloading')
                        if phase == 'unloading':
                            moved = _move_one_container_from_ship_to_wharf(connection, ship_id)
                            if not moved:
                                LIVE_DEMO_STATE['phase'] = 'loading'
                        else:
                            moved = _move_one_container_from_wharf_to_ship(connection, ship_id)
                            if not moved:
                                LIVE_DEMO_STATE['phase'] = 'unloading'
            except (sqlite3.OperationalError, sqlite3.IntegrityError):
                pass

            stop_event.wait(update_interval_s)

    writer_thread = threading.Thread(target=_writer_loop, daemon=True, name='live-demo-writer')
    writer_thread.start()

    LIVE_DEMO_STATE['thread'] = writer_thread
    LIVE_DEMO_STATE['stop_event'] = stop_event

    print(
        f'Live demo writer started for ship {ship_id} (updates every {update_interval_s}s). '
        'Use stop_live_demo_writer() or the widget stop button to stop.'
    )

print('Live demo writer functions loaded. Use the Start/Stop buttons in cell 13.')

Live demo writer functions loaded. Use the Start/Stop buttons in cell 13.


This second cell actually defines the widget.

You can select a ship, and start/stop the background thread to update the data.
If you change ships while the demo is running you'll see the load/unload operation continue on the new ship.

In [8]:
import io
import random
import threading
import contextlib
from datetime import timedelta
import ipywidgets as widgets
from tornado.ioloop import PeriodicCallback

DASHBOARD_STATE = {
    'refresh_callback': None,
    'demo_start_ts': None,
}

def _format_elapsed(seconds):
    return str(timedelta(seconds=int(max(0, seconds))))

def build_visualization_dashboard(default_ship_id=None, refresh_interval_s=0.5):
    ship_ids = list_ship_ids()
    if not ship_ids:
        print('No ships available. Initialize the visualization database first.')
        return

    if default_ship_id is None or default_ship_id not in ship_ids:
        default_ship_id = ship_ids[0]

    ship_dropdown = widgets.Dropdown(
        options=ship_ids,
        value=default_ship_id,
        description='Ship:',
        layout=widgets.Layout(width='200px'),
    )

    start_button = widgets.Button(
        description='Start Demo',
        button_style='success',
        icon='play',
        tooltip='Start live loading/unloading demo',
    )

    stop_button = widgets.Button(
        description='Stop Demo',
        button_style='warning',
        icon='stop',
        tooltip='Stop live loading/unloading demo',
    )

    status_html = widgets.HTML(value='<b>Demo status:</b> stopped')
    timer_html = widgets.HTML(value='<b>Elapsed:</b> 00:00:00')

    attitude_out = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='8px'))
    wharf_out = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='8px'))
    ship_out = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='8px'))

    controls = widgets.HBox([ship_dropdown, start_button, stop_button])
    header = widgets.HBox([status_html, timer_html], layout=widgets.Layout(justify_content='space-between'))
    dashboard = widgets.VBox([controls, header, attitude_out, wharf_out, ship_out])

    def _render_to_output(output_widget, render_fn, *args):
        buffer = io.StringIO()
        with contextlib.redirect_stdout(buffer):
            render_fn(*args)
        with output_widget:
            clear_output(wait=True)
            print(buffer.getvalue(), end='')

    def _refresh_once():
        selected_ship = ship_dropdown.value
        _render_to_output(attitude_out, render_ship_attitude, selected_ship)
        _render_to_output(wharf_out, render_wharf_grid)
        _render_to_output(ship_out, render_ship_grid, selected_ship)

        thread = LIVE_DEMO_STATE.get('thread')
        running = thread is not None and thread.is_alive()
        if running:
            status_html.value = '<b>Demo status:</b> running'
            if DASHBOARD_STATE.get('demo_start_ts') is None:
                DASHBOARD_STATE['demo_start_ts'] = time.time()
            elapsed = time.time() - DASHBOARD_STATE['demo_start_ts']
            timer_html.value = f'<b>Elapsed:</b> {_format_elapsed(elapsed)}'
        else:
            status_html.value = '<b>Demo status:</b> stopped'
            DASHBOARD_STATE['demo_start_ts'] = None
            timer_html.value = '<b>Elapsed:</b> 00:00:00'

    def _start_refresh_loop():
        old_callback = DASHBOARD_STATE.get('refresh_callback')
        if old_callback is not None:
            old_callback.stop()

        interval_ms = max(50, int(refresh_interval_s * 1000))

        def _safe_refresh():
            try:
                _refresh_once()
            except Exception as error:
                with attitude_out:
                    clear_output(wait=True)
                    print(f'Dashboard refresh error: {error}')

        refresh_callback = PeriodicCallback(_safe_refresh, interval_ms)
        refresh_callback.start()
        DASHBOARD_STATE['refresh_callback'] = refresh_callback

    def _on_start_clicked(_):
        selected_ship = ship_dropdown.value
        start_live_demo_writer(ship_id=selected_ship, update_interval_s=0.5)
        DASHBOARD_STATE['demo_start_ts'] = time.time()
        _refresh_once()

    def _on_stop_clicked(_):
        stop_live_demo_writer()
        DASHBOARD_STATE['demo_start_ts'] = None
        _refresh_once()

    def _on_ship_changed(change):
        thread = LIVE_DEMO_STATE.get('thread')
        running = thread is not None and thread.is_alive()
        if running:
            selected_ship = ship_dropdown.value
            start_live_demo_writer(ship_id=selected_ship, update_interval_s=0.5)
        if change.get('name') == 'value':
            _refresh_once()

    start_button.on_click(_on_start_clicked)
    stop_button.on_click(_on_stop_clicked)
    ship_dropdown.observe(_on_ship_changed)

    _refresh_once()
    _start_refresh_loop()

    return dashboard

display(build_visualization_dashboard())